# `infra/`: seeds, configs and run records (walkthrough)

**Owner:** Aya, ML Infrastructure and Evaluation Engineer. **Branch:** `team/aya-ml-infrastructure-evaluation`.

This notebook explains the scripts in `infra/`, the small Python package every training role imports so that all experiments are configured, seeded, recorded and summarised **the same way**. It follows the formats Hazem defined (the shared config, the example config, the run record in the Phase 1 examples). It adds no new rules of its own to them.

**What this notebook does not do.** It trains nothing, evaluates nothing, reads no dataset files, and writes nothing inside the repo. Everything it saves goes to a temporary folder.

### How to read the numbers

| Symbol | Meaning |
|---|---|
| ✅ | A real value, read from `shared_config.json` when you run the notebook. |
| 📋 | An example value copied from the project documents (Hazem's example config, the model selection report). |
| 🧪 | **Synthetic.** Made up only to show a format. It is **not** a result of any model. Never quote it. |

### Contents
1. Setup
2. The shared config
3. Seeds
4. Mean and standard deviation across seeds
5. The experiment config
6. The run record
7. A template for your own experiment
8. Rules, and what is still to come

## 1. Setup

The scripts live in `infra/` at the repo root. To use them from any notebook, the repo root must be on `sys.path`. The cell below finds the repo root the same way Hazem's training skeleton does (the nearest parent folder that has a `.git`), and adds it to `sys.path`.

It also prints the Python version and the package versions. The team uses **Python 3.11** (see `requirements/README.md`). `torch` is optional for `infra/`: Aya's environment does not install it, and the code works with or without it.

In [1]:
import importlib.metadata
import json
import platform
import sys
import tempfile
from pathlib import Path


def find_repo_root(start):
    p = Path(start).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Repo root not found. Run this notebook from inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Everything this notebook saves goes into one temporary folder, never into the repo.
_TMP = tempfile.TemporaryDirectory()
WORK = Path(_TMP.name)


def fresh_dir():
    """A new empty temporary folder. Re-running a cell never hits 'file already exists'."""
    return Path(tempfile.mkdtemp(dir=WORK))


print("Python      :", platform.python_version())
print("Repo folder :", REPO_ROOT.name)
for name in ("numpy", "scikit-learn", "pandas", "pytest", "torch"):
    try:
        print(f"{name:<12}:", importlib.metadata.version(name))
    except importlib.metadata.PackageNotFoundError:
        print(f"{name:<12}: not installed")

Python      : 3.11.9
Repo folder : svr
numpy       : 1.26.4
scikit-learn: 1.9.1
pandas      : 2.1.4
pytest      : 8.3.4
torch       : not installed


In [2]:
from dataclasses import replace

import numpy as np

from infra.experiment_config import (
    ExperimentConfig,
    ExperimentConfigError,
    changed_fields,
    load_experiment_config,
    save_experiment_config,
)
from infra.run_record import (
    RunRecordError,
    build_run_record,
    load_run_record,
    make_seed_result,
    save_run_record,
)
from infra.seeds import run_over_seeds, set_global_seed, summarize, summarize_metrics
from infra.shared_config import SharedConfigError, load_shared_config

print("infra imported OK")

infra imported OK


## 2. The shared config (defined by Hazem)

`notebooks/training/shared_config.json` holds the values the whole team shares: the split seed, the five experiment seeds, the maximum sequence length, the dataset version and the row counts.

**Rule: `infra/` only reads this file. It never writes it and never keeps its own copy of the values.** If Hazem changes a value, everyone gets the change on the next run. That is why the cells below read the numbers from the file instead of typing them.

`load_shared_config` also **validates** the file, so a wrong value stops the run immediately instead of silently changing results.

In [3]:
CANDIDATES = [
    REPO_ROOT / "notebooks" / "training" / "shared_config.json",
    REPO_ROOT.parent / "svr-data" / "notebooks" / "training" / "shared_config.json",
    REPO_ROOT.parent / "smart-voicemail-router" / "notebooks" / "training" / "shared_config.json",
]
WHERE = ["inside this repo", "sibling folder svr-data", "sibling folder smart-voicemail-router"]

found = [(label, p) for label, p in zip(WHERE, CANDIDATES) if p.is_file()]
if not found:
    raise FileNotFoundError(
        "shared_config.json not found. Looked in: " + "; ".join(WHERE)
        + ". It lives at notebooks/training/shared_config.json on the data branch."
    )
where, SHARED_CONFIG_PATH = found[0]
shared = load_shared_config(SHARED_CONFIG_PATH)

print("shared_config.json found:", where)
print("loaded and validated OK  ✅")
print()
for key, value in shared.items():
    print(f"  {key:<17}{value!r}")

shared_config.json found: sibling folder smart-voicemail-router
loaded and validated OK  ✅

  split_seed       42
  experiment_seeds [42, 1337, 2023, 7, 99]
  max_seq_len      128
  split_ratio      '70/20/10'
  dataset_version  'v2'
  dataset          'svr-voicemail-corpus'
  train_rows       3575
  val_rows         1003
  test_rows        518


**Checking the numbers against each other.** The config states the split ratio as text and the row counts as numbers. The cell below computes the percentages from the row counts and compares them with the stated ratio. Nothing is typed by hand.

In [4]:
rows = {s: shared[f"{s}_rows"] for s in ("train", "val", "test")}
total = sum(rows.values())
print(f"total clean rows ✅ : {total}")
print()
print(f"{'split':<7}{'rows':>7}{'share':>10}")
for split, n in rows.items():
    print(f"{split:<7}{n:>7}{100 * n / total:>9.2f}%")

try:
    stated = [int(x) for x in shared["split_ratio"].split("/")]
    rounded = [round(100 * n / total) for n in rows.values()]
    print()
    print("split_ratio in the config      :", shared["split_ratio"])
    print("rounded from the row counts    :", "/".join(map(str, rounded)))
    print("they agree                     :", stated == rounded)
except ValueError:
    print("split_ratio is not in the form 'a/b/c', so it was not compared.")

total clean rows ✅ : 5096

split     rows     share
train     3575    70.15%
val       1003    19.68%
test       518    10.16%

split_ratio in the config      : 70/20/10
rounded from the row counts    : 70/20/10
they agree                     : True


**What the loader rejects.** The cell below builds a few broken copies of the real config (in memory, in a temporary folder) and shows what the loader says about each. The first case is an untouched copy and must be accepted.

In [5]:
def try_load(label, data):
    path = fresh_dir() / "shared_config.json"
    path.write_text(json.dumps(data), encoding="utf-8")
    try:
        load_shared_config(path)
        print(f"{label:<22}: accepted")
    except SharedConfigError as exc:
        print(f"{label:<22}: REJECTED -> {exc}")


broken = {
    "untouched copy": dict(shared),
    "missing test_rows": {k: v for k, v in shared.items() if k != "test_rows"},
    "duplicate seeds": dict(shared, experiment_seeds=[42, 42, 7]),
    "a seed as text": dict(shared, experiment_seeds=[42, "1337"]),
    "split_seed = True": dict(shared, split_seed=True),
}
for label, data in broken.items():
    try_load(label, data)

untouched copy        : accepted
missing test_rows     : REJECTED -> shared config is missing keys: ['test_rows']
duplicate seeds       : REJECTED -> experiment_seeds has duplicates: [42, 42, 7]
a seed as text        : REJECTED -> experiment_seeds must be a non-empty list of integers, got [42, '1337']
split_seed = True     : REJECTED -> split_seed must be an integer, got True


## 3. Seeds

**Why five seeds?** Fine-tuning a model on a small dataset is noisy: the same setup with a different seed gives a different score, and the rare classes (for example `at_risk` churn) vary the most. One run can be lucky. The roles guide therefore requires every experiment to run on **all five seeds** and to report the mean and the standard deviation (see also `research/notes/training-stability-mosbach.md`).

**Two different jobs, easy to mix up:**
- `split_seed` decides how the data was split into train, validation and test. It is fixed. The data is never re-split.
- `experiment_seeds` are the seeds for training runs. Every experiment runs once per seed.

The number 42 appears in both, but the two have nothing to do with each other.

In [6]:
seeds = shared["experiment_seeds"]
print("split_seed       ✅ :", shared["split_seed"], "(used once, to split the data)")
print("experiment_seeds ✅ :", seeds, "(one training run per seed)")

split_seed       ✅ : 42 (used once, to split the data)
experiment_seeds ✅ : [42, 1337, 2023, 7, 99] (one training run per seed)


**`set_global_seed(seed)`** seeds Python's `random`, NumPy and, if it is installed, PyTorch. The same seed must give the same numbers, and a different seed a different one.

In [7]:
import random


def draw():
    return round(random.random(), 6), round(float(np.random.rand()), 6)


s_a, s_b = seeds[0], seeds[1]
set_global_seed(s_a); first = draw()
set_global_seed(s_a); again = draw()
set_global_seed(s_b); other = draw()

print(f"seed {s_a:<5}:", first)
print(f"seed {s_a:<5}:", again, "  (same seed again)")
print(f"seed {s_b:<5}:", other)
print()
print("same seed gives the same numbers      :", first == again)
print("a different seed gives other numbers  :", first != other)

seed 42   : (0.639427, 0.37454)
seed 42   : (0.639427, 0.37454)   (same seed again)
seed 1337 : (0.617753, 0.262025)

same seed gives the same numbers      : True
a different seed gives other numbers  : True


**`run_over_seeds(fn, seeds)`** calls `fn(seed)` once per seed, in order, and sets the global seed before each call. The function you pass is where your training and validation code goes. Below it is a toy function that only draws a random number 🧪, to show that running everything twice gives identical results.

Note: setting the global seed is not enough on its own. Whenever a library accepts a seed argument (for example `random_state=seed` in scikit-learn), pass the seed explicitly too.

In [8]:
def toy(seed):
    return round(float(np.random.rand()), 4)


run_1 = run_over_seeds(toy, seeds)
run_2 = run_over_seeds(toy, seeds)

print("🧪 toy function, one result per seed")
for rec in run_1:
    print(f"  seed {rec['seed']:<5} -> {rec['result']}")
print()
print("a second full run gives identical results:", run_1 == run_2)

🧪 toy function, one result per seed
  seed 42    -> 0.3745
  seed 1337  -> 0.262
  seed 2023  -> 0.322
  seed 7     -> 0.0763
  seed 99    -> 0.6723

a second full run gives identical results: True


**PyTorch is optional.** `infra/` must work on machines without `torch`, so it seeds `torch` only when it is installed. Lama's and Sira's environments have it; Aya's does not.

In [9]:
try:
    import torch
except ImportError:
    print("torch is not installed in this environment.")
    print("set_global_seed() skips it, and everything else works.")
else:
    set_global_seed(seeds[-1])
    print(f"torch is installed. After set_global_seed({seeds[-1]}):")
    print("  torch.initial_seed() =", torch.initial_seed())

torch is not installed in this environment.
set_global_seed() skips it, and everything else works.


## 4. Mean and standard deviation across seeds

For each metric, an experiment reports the **mean** and the **standard deviation (std)** over its five seed results. Hazem confirmed that the std is the **sample** standard deviation (`ddof=1`): it divides by `n - 1` (4 for five seeds), not by `n`. This is the right choice for a small sample.

The cell below computes it step by step on five made-up scores 🧪, then checks the result against NumPy and Python's `statistics` module.

In [10]:
import statistics

scores = [0.62, 0.65, 0.60, 0.64, 0.63]  # 🧪 synthetic
n = len(scores)
mean = sum(scores) / n

print(f"n = {n}, mean = sum / n = {mean:.4f}")
print()
print(f"{'score':>7}{'deviation':>12}{'squared':>12}")
squares = []
for s in scores:
    d = s - mean
    squares.append(d * d)
    print(f"{s:>7.2f}{d:>12.4f}{d * d:>12.6f}")

sum_sq = sum(squares)
variance = sum_sq / (n - 1)
std_by_hand = variance ** 0.5
print()
print(f"sum of squares          = {sum_sq:.6f}")
print(f"divide by n - 1 = {n - 1}     = {variance:.6f}   (the sample variance)")
print(f"square root             = {std_by_hand:.6f}   (the sample std)")
print()
ours = summarize(scores, ddof=1)
print("infra.summarize          :", round(ours["std"], 6))
print("numpy, ddof=1            :", round(float(np.std(scores, ddof=1)), 6))
print("statistics.stdev         :", round(statistics.stdev(scores), 6))
print("all four agree           :", abs(std_by_hand - ours["std"]) < 1e-12
      and abs(ours["std"] - statistics.stdev(scores)) < 1e-12)

n = 5, mean = sum / n = 0.6280

  score   deviation     squared
   0.62     -0.0080    0.000064
   0.65      0.0220    0.000484
   0.60     -0.0280    0.000784
   0.64      0.0120    0.000144
   0.63      0.0020    0.000004

sum of squares          = 0.001480
divide by n - 1 = 4     = 0.000370   (the sample variance)
square root             = 0.019235   (the sample std)

infra.summarize          : 0.019235
numpy, ddof=1            : 0.019235
statistics.stdev         : 0.019235
all four agree           : True


**`ddof=1` versus `ddof=0`.** With `ddof=0` the sum of squares is divided by `n` instead of `n - 1`, which gives a smaller number. With five seeds the two differ by a fixed factor, `sqrt(n / (n - 1))`. Every result records the `ddof` it used, so a std is never reported without saying which formula produced it.

In [11]:
s1 = summarize(scores, ddof=1)
s0 = summarize(scores, ddof=0)
factor = (n / (n - 1)) ** 0.5

print(f"{'':<10}{'mean':>9}{'std':>10}{'ddof':>6}{'n':>4}")
print(f"{'ddof=1':<10}{s1['mean']:>9.4f}{s1['std']:>10.5f}{s1['ddof']:>6}{s1['n']:>4}   <- the team's choice")
print(f"{'ddof=0':<10}{s0['mean']:>9.4f}{s0['std']:>10.5f}{s0['ddof']:>6}{s0['n']:>4}")
print()
print(f"std(ddof=1) / std(ddof=0)       = {s1['std'] / s0['std']:.4f}")
print(f"sqrt(n / (n - 1)) for n = {n}      = {factor:.4f}")
print(f"so ddof=1 is {100 * (factor - 1):.1f}% larger than ddof=0 when n = {n}")

               mean       std  ddof   n
ddof=1       0.6280   0.01924     1   5   <- the team's choice
ddof=0       0.6280   0.01720     0   5

std(ddof=1) / std(ddof=0)       = 1.1180
sqrt(n / (n - 1)) for n = 5      = 1.1180
so ddof=1 is 11.8% larger than ddof=0 when n = 5


**`summarize_metrics`** does the same for a whole set of metrics at once. It takes one dictionary of metrics per seed and requires every seed to report the same metric names. This is what the run record uses.

`summarize` also refuses input that would give a meaningless std, for example a single value with `ddof=1`, or a value that is not a finite number.

In [12]:
per_seed_demo = [  # 🧪 synthetic: one dict per seed
    {"metric_a": 0.62, "metric_b": 0.50},
    {"metric_a": 0.65, "metric_b": 0.53},
    {"metric_a": 0.60, "metric_b": 0.49},
    {"metric_a": 0.64, "metric_b": 0.52},
    {"metric_a": 0.63, "metric_b": 0.51},
]
table = summarize_metrics(per_seed_demo, ddof=1)
print(f"{'metric':<10}{'mean':>9}{'std':>10}{'ddof':>6}{'n':>4}")
for name, s in table.items():
    print(f"{name:<10}{s['mean']:>9.4f}{s['std']:>10.5f}{s['ddof']:>6}{s['n']:>4}")

print()
for label, call in {
    "a single value, ddof=1": lambda: summarize([0.7], ddof=1),
    "a NaN value": lambda: summarize([0.7, float("nan")]),
    "seeds with different metrics": lambda: summarize_metrics([{"a": 0.5}, {"b": 0.6}]),
}.items():
    try:
        call()
        print(f"{label:<30}: accepted")
    except ValueError as exc:
        print(f"{label:<30}: REJECTED -> {exc}")

metric         mean       std  ddof   n
metric_a     0.6280   0.01924     1   5
metric_b     0.5100   0.01581     1   5

a single value, ddof=1        : REJECTED -> need more than 1 value(s) to compute std with ddof=1
a NaN value                   : REJECTED -> values must be finite numbers, got nan
seeds with different metrics  : REJECTED -> seeds report different metrics: ['a'] vs ['b']


> **What this module does not do.** It reports the mean and the std. It does not pick a winner. The tie-break rule between EgyBERT and MARBERTv2 is defined in the model selection report and applied by Hazem; it uses these numbers.

## 5. The experiment config

Hazem's example config has these fields: **model, learning rate, batch size, epochs, seed, dataset version, split**. `ExperimentConfig` keeps exactly those. Three details:

1. **The seed is not stored in the config.** One experiment is one config run once per seed, and the seeds come from `shared_config.json`. `for_seed(seed)` returns the full per-run config, with the seed added.
2. **Only the `train` split is accepted for training.** Anything else (`val`, `test`) is rejected, so nobody can train on the test set by mistake.
3. **Fields that do not apply are left as `None`.** The TF-IDF baseline has no learning rate, batch size or epochs. Anything model specific beyond those goes in `other_hyperparameters`.

In [13]:
EXAMPLE = dict(model="EgyBERT", learning_rate=2e-5, batch_size=16, epochs=3)  # 📋 Hazem's example
config = ExperimentConfig(
    dataset_version=shared["dataset_version"],  # ✅ from the shared config
    split="train",
    **EXAMPLE,
)

print("The experiment config (the same for all five seeds):")
print(json.dumps(config.to_dict(), indent=2))
print()
print(f"The config of one run, for seed {seeds[0]} (this is the shape in Hazem's example):")
print(json.dumps(config.for_seed(seeds[0]), indent=2))

The experiment config (the same for all five seeds):
{
  "model": "EgyBERT",
  "dataset_version": "v2",
  "split": "train",
  "learning_rate": 2e-05,
  "batch_size": 16,
  "epochs": 3,
  "other_hyperparameters": {}
}

The config of one run, for seed 42 (this is the shape in Hazem's example):
{
  "model": "EgyBERT",
  "dataset_version": "v2",
  "split": "train",
  "learning_rate": 2e-05,
  "batch_size": 16,
  "epochs": 3,
  "other_hyperparameters": {},
  "seed": 42
}


In [14]:
baseline = ExperimentConfig(
    model="tfidf-logreg",
    dataset_version=shared["dataset_version"],
    other_hyperparameters={"max_features": 20000, "C": 1.0},  # 🧪 illustrative values only
)
print("A non-neural model leaves the neural fields empty:")
print(json.dumps(baseline.to_dict(), indent=2))

A non-neural model leaves the neural fields empty:
{
  "model": "tfidf-logreg",
  "dataset_version": "v2",
  "split": "train",
  "learning_rate": null,
  "batch_size": null,
  "epochs": null,
  "other_hyperparameters": {
    "max_features": 20000,
    "C": 1.0
  }
}


**What the config rejects.** Each case below is a mistake that would otherwise slip into a run unnoticed.

In [15]:
def show(label, make):
    try:
        make()
        print(f"{label:<40}: accepted")
    except ExperimentConfigError as exc:
        print(f"{label:<40}: REJECTED -> {exc}")


ds = shared["dataset_version"]
show("split = 'test'", lambda: ExperimentConfig(model="m", dataset_version=ds, split="test"))
show("split = 'val'", lambda: ExperimentConfig(model="m", dataset_version=ds, split="val"))
show("learning_rate = 0", lambda: ExperimentConfig(model="m", dataset_version=ds, learning_rate=0))
show("batch_size = 1.5", lambda: ExperimentConfig(model="m", dataset_version=ds, batch_size=1.5))
show("seed inside the other hyperparameters",
     lambda: ExperimentConfig(model="m", dataset_version=ds, other_hyperparameters={"seed": 1}))
show("'seed' in a config file",
     lambda: ExperimentConfig.from_dict({"model": "m", "dataset_version": ds, "seed": 42}))
show("an unknown field (typo 'lr')",
     lambda: ExperimentConfig.from_dict({"model": "m", "dataset_version": ds, "lr": 1e-3}))

split = 'test'                          : REJECTED -> split must be one of ('train',), got 'test'
split = 'val'                           : REJECTED -> split must be one of ('train',), got 'val'
learning_rate = 0                       : REJECTED -> learning_rate must be a positive number or None, got 0
batch_size = 1.5                        : REJECTED -> batch_size must be a positive integer or None, got 1.5
seed inside the other hyperparameters   : REJECTED -> other_hyperparameters must not contain ['seed']; use the named fields (the seed comes from shared_config.json)
'seed' in a config file                 : REJECTED -> do not put 'seed' in the experiment config: seeds come from shared_config.json, use for_seed(seed)
an unknown field (typo 'lr')            : REJECTED -> unknown config fields: ['lr']


**`changed_fields(a, b)`** lists what differs between two configs. The run record uses it to enforce the rule **change one factor at a time**: if you compare a run against its reference and more than one thing changed, you cannot tell which one made the difference.

In [16]:
reference = config
print("same config            :", changed_fields(reference, replace(config)))
print("learning rate only     :", changed_fields(reference, replace(config, learning_rate=3e-5)))
print("model only             :", changed_fields(reference, replace(config, model="MARBERTv2")))
print("two things at once     :", changed_fields(reference, replace(config, learning_rate=3e-5, batch_size=8)))
print("a hyperparameter inside:", changed_fields(
    replace(config, other_hyperparameters={"warmup_ratio": 0.1}),
    replace(config, other_hyperparameters={"warmup_ratio": 0.06}),
))

same config            : []
learning rate only     : ['learning_rate']
model only             : ['model']
two things at once     : ['batch_size', 'learning_rate']
a hyperparameter inside: ['other_hyperparameters.warmup_ratio']


**Saving and loading.** A config file is plain JSON. `save_experiment_config` refuses to overwrite an existing file, so an old experiment's config can never change silently, and `load_experiment_config` can check the dataset version against the shared config at the same time.

In [17]:
folder = fresh_dir()
path = folder / "exp_config.json"
save_experiment_config(config, path)
print("saved:", path.name)
print("loaded config equals the saved one:", load_experiment_config(path, shared_config=shared) == config)

try:
    save_experiment_config(replace(config, epochs=5), path)
except FileExistsError as exc:
    print("saving a different config to the same file -> REFUSED:", path.name, "already exists")

old = fresh_dir() / "old.json"
save_experiment_config(replace(config, dataset_version="v-old"), old)
try:
    load_experiment_config(old, shared_config=shared)
except ExperimentConfigError as exc:
    print("a config for another dataset version -> REJECTED:", exc)

saved: exp_config.json
loaded config equals the saved one: True
saving a different config to the same file -> REFUSED: exp_config.json already exists
a config for another dataset version -> REJECTED: dataset_version 'v-old' does not match shared_config 'v2'


## 6. The run record

Every experiment has one run record, a single JSON file. Hazem's version has six parts:

| Part | Meaning |
|---|---|
| **run ID** | The experiment's name, for example `001`. |
| **hypothesis** | What you expect, in one sentence. |
| **changed factor** | The one thing that differs from the reference run. |
| **held-constant conditions** | Everything else, which stayed the same. |
| **measures** | The macro-F1 for each of the four fields, per seed, with the mean and std. |
| **artifacts** | Where the checkpoints and prediction files are. |

The record also stores the dataset version and row counts, the environment (Python, packages, device) and a timestamp, so any number can be traced and reproduced.

**Computed, not typed.** `held_constant` is derived from the config. The mean and std are computed from the per-seed results. The check that exactly one factor changed compares the config with the reference config. So these parts cannot be inconsistent with each other.

The metric names are the ones in Hazem's `make_result_record`: `complaint_category_macro_f1`, `complaint_urgency_macro_f1`, `customer_state_macro_f1`, `churn_risk_macro_f1`.

### 6.1 A complete example

> **🧪 Every metric value in this section is synthetic.** The numbers are deliberately made up and unrelated to any real model. The hypothesis below is Hazem's example text; the made-up numbers prove nothing about it.

In [18]:
SYNTHETIC = {  # 🧪 one value per seed, in the order of the shared config's seeds
    "complaint_category_macro_f1": [0.60, 0.62, 0.58, 0.61, 0.59],
    "complaint_urgency_macro_f1": [0.50, 0.52, 0.48, 0.51, 0.49],
    "customer_state_macro_f1": [0.40, 0.42, 0.38, 0.41, 0.39],
    "churn_risk_macro_f1": [0.30, 0.34, 0.26, 0.32, 0.28],
}
assert len(seeds) == 5, "this example has five values per metric; adjust it if the seed list changes"

per_seed = [
    make_seed_result("001", config, "val", seed,
                     {name: values[i] for name, values in SYNTHETIC.items()})
    for i, seed in enumerate(seeds)
]
print("One seed's result (the shape of make_result_record in the training skeleton):")
print(json.dumps(per_seed[0], indent=2))

One seed's result (the shape of make_result_record in the training skeleton):
{
  "run_id": "001",
  "model": "EgyBERT",
  "split": "val",
  "seed": 42,
  "metrics": {
    "complaint_category_macro_f1": 0.6,
    "complaint_urgency_macro_f1": 0.5,
    "customer_state_macro_f1": 0.4,
    "churn_risk_macro_f1": 0.3
  }
}


In [19]:
marbert = replace(config, model="MARBERTv2")  # the reference run: the same config with the other model

record = build_run_record(
    run_id="001",
    hypothesis="EgyBERT beats MARBERTv2 on complaint_category",
    config=config,
    shared_config=shared,
    per_seed=per_seed,
    evaluated_split="val",
    artifacts={
        "checkpoints": [f"runs/exp_001/checkpoints/seed_{s}" for s in seeds],
        "predictions": ["runs/exp_001/predictions_val.json"],
    },
    device="cpu",
    reference_config=marbert,
    changed_factor="model",
    reference_run_id="002",  # the run that produced the MARBERTv2 numbers
)
print("record built. Top-level parts:")
for key in record:
    print("  -", key)

record built. Top-level parts:
  - schema_version
  - run_id
  - hypothesis
  - changed_factor
  - held_constant
  - reference_run_id
  - config
  - dataset
  - evaluated_split
  - per_seed
  - aggregate
  - artifacts
  - environment
  - created_utc


In [20]:
print("run_id          :", record["run_id"])
print("hypothesis      :", record["hypothesis"])
print("changed_factor  :", record["changed_factor"])
print("held_constant   :", record["held_constant"])
print("evaluated_split :", record["evaluated_split"], "(the split the numbers come from)")
print("dataset         :", record["dataset"], "  <- ✅ from the shared config")
print("environment     :", record["environment"])
print()
print("`held_constant` was derived: the config minus the changed factor.")
print("Note that `split` in the config is the TRAINING split; `evaluated_split` is where the numbers come from.")

run_id          : 001
hypothesis      : EgyBERT beats MARBERTv2 on complaint_category
changed_factor  : model
held_constant   : {'dataset_version': 'v2', 'split': 'train', 'learning_rate': 2e-05, 'batch_size': 16, 'epochs': 3}
evaluated_split : val (the split the numbers come from)
dataset         : {'version': 'v2', 'rows': {'train': 3575, 'val': 1003, 'test': 518}}   <- ✅ from the shared config
environment     : {'python': '3.11.9', 'platform': 'Windows-10-10.0.26200-SP0', 'device': 'cpu', 'packages': {'numpy': '1.26.4', 'pandas': '2.1.4', 'scikit-learn': '1.9.1'}}

`held_constant` was derived: the config minus the changed factor.
Note that `split` in the config is the TRAINING split; `evaluated_split` is where the numbers come from.


In [21]:
print("aggregate (computed from the five seeds, ddof recorded):")
print(f"  {'metric':<30}{'mean':>8}{'std':>9}{'ddof':>6}{'n':>4}")
for name, s in record["aggregate"].items():
    print(f"  {name:<30}{s['mean']:>8.4f}{s['std']:>9.5f}{s['ddof']:>6}{s['n']:>4}")

print()
print("per seed:")
names = list(SYNTHETIC)
print("  " + f"{'seed':<7}" + "".join(f"{n.replace('_macro_f1', ''):>22}" for n in names))
for r in record["per_seed"]:
    print("  " + f"{r['seed']:<7}" + "".join(f"{r['metrics'][n]:>22.2f}" for n in names))

aggregate (computed from the five seeds, ddof recorded):
  metric                            mean      std  ddof   n
  churn_risk_macro_f1             0.3000  0.03162     1   5
  complaint_category_macro_f1     0.6000  0.01581     1   5
  complaint_urgency_macro_f1      0.5000  0.01581     1   5
  customer_state_macro_f1         0.4000  0.01581     1   5

per seed:
  seed       complaint_category     complaint_urgency        customer_state            churn_risk
  42                       0.60                  0.50                  0.40                  0.30
  1337                     0.62                  0.52                  0.42                  0.34
  2023                     0.58                  0.48                  0.38                  0.26
  7                        0.61                  0.51                  0.41                  0.32
  99                       0.59                  0.49                  0.39                  0.28


### 6.2 What the run record rejects

Each case below changes one thing in the correct example and shows the error. These are the mistakes the record is designed to catch before they reach a report.

In [22]:
def build_with(**changes):
    args = dict(
        run_id="001",
        hypothesis="EgyBERT beats MARBERTv2 on complaint_category",
        config=config,
        shared_config=shared,
        per_seed=per_seed,
        evaluated_split="val",
        artifacts={"predictions": ["runs/exp_001/predictions_val.json"]},
        device="cpu",
        reference_config=marbert,
        changed_factor="model",
    )
    args.update(changes)
    return build_run_record(**args)


def refuse(label, **changes):
    try:
        build_with(**changes)
        print(f"{label:<38}: accepted")
    except (RunRecordError, ExperimentConfigError) as exc:
        print(f"{label:<38}: REJECTED -> {exc}")


refuse("the correct example")
refuse("only four seeds", per_seed=per_seed[:-1])
refuse("seeds in a different order", per_seed=[per_seed[1], per_seed[0], *per_seed[2:]])
refuse("two factors changed", reference_config=replace(marbert, learning_rate=3e-5))
refuse("nothing changed", reference_config=config)
refuse("changed_factor names the wrong thing", changed_factor="learning_rate")
refuse("a factor but no reference config", reference_config=None)

missing_metric = [
    {**r, "metrics": {k: v for k, v in r["metrics"].items() if k != "churn_risk_macro_f1"}}
    for r in per_seed
]
refuse("a required metric is missing", per_seed=missing_metric)

too_high = [
    {**r, "metrics": {**r["metrics"], "customer_state_macro_f1": 1.2}} if i == 0 else r
    for i, r in enumerate(per_seed)
]
refuse("a macro-F1 above 1", per_seed=too_high)
refuse("no device written", device="")
refuse("an absolute artifact path", artifacts={"predictions": ["C:/Users/someone/pred.json"]})
refuse("an artifact name with 'transcript'", artifacts={"predictions": ["runs/exp_001/val_transcript_check.json"]})

the correct example                   : accepted
only four seeds                       : REJECTED -> every experiment must run all seeds in order: expected [42, 1337, 2023, 7, 99], got [42, 1337, 2023, 7]
seeds in a different order            : REJECTED -> every experiment must run all seeds in order: expected [42, 1337, 2023, 7, 99], got [1337, 42, 2023, 7, 99]
two factors changed                   : REJECTED -> more than one factor changed: ['learning_rate', 'model']
nothing changed                       : REJECTED -> nothing changed compared to the reference config
changed_factor names the wrong thing  : REJECTED -> changed_factor is 'learning_rate' but the configs differ in 'model'
a factor but no reference config      : REJECTED -> changed_factor needs a reference_config to check it against
a required metric is missing          : REJECTED -> seed 42: missing metrics ['churn_risk_macro_f1']
a macro-F1 above 1                    : REJECTED -> seed 42: customer_state_macro_f1 must be

### 6.3 Saving and loading

A run record is saved as `runs/exp_<run_id>/run_record.json` and **never overwritten**. The cell below saves into a temporary folder, shows the layout and the start of the file, and loads it back.

In [23]:
runs_dir = fresh_dir()
saved = save_run_record(record, runs_dir)
print("saved as:", saved.relative_to(runs_dir).as_posix())
print("loads back identical:", load_run_record(saved) == record)

try:
    save_run_record(record, runs_dir)
except FileExistsError:
    print("saving the same run again -> REFUSED (never overwritten)")

print()
print("first 30 lines of the file:")
print("\n".join(saved.read_text(encoding="utf-8").splitlines()[:30]))

saved as: exp_001/run_record.json
loads back identical: True
saving the same run again -> REFUSED (never overwritten)

first 30 lines of the file:
{
  "schema_version": 1,
  "run_id": "001",
  "hypothesis": "EgyBERT beats MARBERTv2 on complaint_category",
  "changed_factor": "model",
  "held_constant": {
    "dataset_version": "v2",
    "split": "train",
    "learning_rate": 2e-05,
    "batch_size": 16,
    "epochs": 3
  },
  "reference_run_id": "002",
  "config": {
    "model": "EgyBERT",
    "dataset_version": "v2",
    "split": "train",
    "learning_rate": 2e-05,
    "batch_size": 16,
    "epochs": 3,
    "other_hyperparameters": {}
  },
  "dataset": {
    "version": "v2",
    "rows": {
      "train": 3575,
      "val": 1003,
      "test": 518
    }
  },


**What goes into Git, and what does not.** The team `.gitignore` decides:

| File | In Git? | Why |
|---|---|---|
| `run_record.json`, other `.json` result files | ✅ yes | Hazem reviews them. |
| checkpoints (`checkpoints/`, `*.pt`, `*.safetensors`, `*.pkl`, ...) | ❌ no | Large. The record stores only their **relative path**. |
| `.csv` files | ❌ no | Ignored. Save predictions as JSON. |
| `*transcript*.json`, `*transcript*.txt`, `*transcript*.csv` | ❌ no | Ignored, and may hold raw text. The record rejects any artifact name containing `transcript`. |

Keep raw voicemail text out of the files you commit. Predictions refer to a sample by `split` and `audio_file` (sample names repeat between splits, so use both), never by copying its text.

## 7. A template for your own experiment

This is the whole flow. **Replace the body of `train_and_eval(seed)`** with your training and validation code. It must return the validation macro-F1 for each of the four fields. Everything around it stays the same.

The example is a learning-rate sweep: the reference is the run from section 6 (learning rate 2e-5), and the two sweep values are the ones in the roles guide (1e-5 and 3e-5, one changed factor each). The metrics are **🧪 synthetic**.

In [24]:
SWEEP_LRS = [1e-5, 3e-5]  # 📋 the roles guide sweeps 1e-5, 2e-5, 3e-5 (2e-5 is the reference)
BASE = {name: values[0] for name, values in SYNTHETIC.items()}  # 🧪


def make_train_and_eval(run_config):
    def train_and_eval(seed):
        # <- your training and validation code goes here.
        # Pass `seed` to anything that accepts one (for example random_state=seed).
        rng = np.random.default_rng(seed)
        shift = (run_config.learning_rate - 2e-5) * 1000  # 🧪 an invented lr effect
        return {
            name: float(np.clip(base + shift + rng.normal(0, 0.01), 0, 1))
            for name, base in BASE.items()
        }
    return train_and_eval


sweep_dir = fresh_dir()
for i, lr in enumerate(SWEEP_LRS):
    run_id = f"{i + 3:03d}"
    run_config = replace(config, learning_rate=lr)                               # 1. one config
    results = run_over_seeds(make_train_and_eval(run_config), shared["experiment_seeds"])  # 2. every seed
    seed_results = [                                                              # 3. the per-seed results
        make_seed_result(run_id, run_config, "val", r["seed"], r["result"]) for r in results
    ]
    rec = build_run_record(                                                       # 4. the record
        run_id=run_id,
        hypothesis=f"learning rate {lr:g} changes validation macro-F1 compared with 2e-5",
        config=run_config,
        shared_config=shared,
        per_seed=seed_results,
        evaluated_split="val",
        artifacts={"predictions": [f"runs/exp_{run_id}/predictions_val.json"]},
        device="cpu",
        reference_config=config,
        changed_factor="learning_rate",
        reference_run_id="001",
    )
    save_run_record(rec, sweep_dir)                                               # 5. save it
    print(f"run {run_id} saved ({rec['changed_factor']} = {lr:g})")

run 003 saved (learning_rate = 1e-05)
run 004 saved (learning_rate = 3e-05)


In [25]:
print("🧪 synthetic numbers, shown only to illustrate how runs are read side by side")
print(f"{'run':<5}{'changed factor':<17}{'value':>9}   complaint_category_macro_f1 (mean ± std, ddof=1, n=5)")
for run_dir in sorted(sweep_dir.glob("exp_*")):
    rec = load_run_record(run_dir / "run_record.json")
    agg = rec["aggregate"]["complaint_category_macro_f1"]
    print(f"{rec['run_id']:<5}{rec['changed_factor']:<17}{rec['config']['learning_rate']:>9g}   "
          f"{agg['mean']:.4f} ± {agg['std']:.4f}")

🧪 synthetic numbers, shown only to illustrate how runs are read side by side
run  changed factor       value   complaint_category_macro_f1 (mean ± std, ddof=1, n=5)
003  learning_rate        1e-05   0.5921 ± 0.0025
004  learning_rate        3e-05   0.6121 ± 0.0025


## 8. Rules, and what is still to come

**Rules from the roles guide that these scripts enforce**

| Rule | Where it is enforced |
|---|---|
| Every experiment runs on all five seeds. | `build_run_record` rejects a run without the exact seeds of the shared config, in order. |
| Change one factor at a time. | `changed_fields` plus `build_run_record`. |
| Nobody reports a result without a run record. | The record is the only way to get a result into the repo in the agreed format. |
| Never train on the test set. | `ExperimentConfig` accepts only `split="train"`. |
| The dataset version is the team's. | The config and the record are checked against the shared config. |
| No data, audio or raw text in Git. | Records hold paths and numbers only; the artifact checks reject absolute paths and `transcript` names. |

**What these scripts do not do (yet)**
- They do not compute metrics. The **evaluation harness** (macro-F1, accuracy and the confusion matrix per field, on mock predictions first) is the next piece.
- They do not train anything. The **TF-IDF baseline** comes after the harness.
- They do not yet enforce that the **test set is evaluated exactly once**. That guard comes with the harness and the final evaluation.
- They do not choose a winner. That is Hazem's decision, using the numbers in the run records.
- `mlflow` is optional and not used. The JSON run record is the source of truth.

**Questions or changes:** ask Aya. If you need a field that is not in the config or the record, ask before adding your own format, so everyone keeps using the same one.